# Prompt Engineering Techniques

One running example, built up one technique at a time:

1. Clear and Direct
2. Being Specific
3. Structure with XML Tags
4. Providing Examples (Few-Shot)

**The task:** John has written a vacation-request email in French to his manager. We want Claude to translate it into English, in a specific tone, formatted as a proper email. We'll write the same request four times, adding one prompt engineering technique at each step, so you can see exactly what each technique changes and why.


In [1]:
# Setup: load environment variables and create the client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text


In [2]:
# The employee's email — the input data we'll be working with throughout
employee_email = """
Objet : Demande de Congé

Cher [Nom du Responsable],

Je m'appelle John et je travaille au sein de la société XYZ.
J'aimerais solliciter une demande de congé pour partir en vacances.
Serait-il possible de discuter des dates qui conviendraient le mieux pour l'équipe et l'entreprise?

Je vous remercie par avance pour votre compréhension et j'attends votre retour.

Cordialement,
John
"""


## 1. Clear and Direct

**Definition:** Being clear and direct means stating exactly what you want Claude to do, using simple, unambiguous language — no room for Claude to guess your intent.

Let's start with the most basic version: just ask Claude to translate the email. Notice this prompt doesn't say *how* to translate it, what tone to use, or what the output should look like — it's clear about the *action* (translate) but says nothing else.


In [3]:
prompt_v1 = f"""Translate this email into English.

{employee_email}
"""

messages = []
add_user_message(messages, prompt_v1)
response_v1 = chat(messages)
print(response_v1)


**Subject:** Leave Request

Dear [Manager's Name],

My name is John, and I work at XYZ company.
I would like to request leave to go on vacation.
Would it be possible to discuss which dates would be most suitable for the team and the company?

Thank you in advance for your understanding, and I look forward to your reply.

Best regards,
John


This works, but Claude had to guess everything else: what tone to use, how formal to be, whether to keep the email format. Different runs could give noticeably different results. Let's fix that by being specific.


## 2. Being Specific

**Definition:** Being specific means giving Claude concrete guidelines about what the output should look like — tone, length, structure, or particular elements to include — instead of leaving it to Claude's interpretation.

We add a short list of guidelines: the target tone, and a reminder to keep it in proper email format. This directly targets the ambiguity we saw in v1.


In [4]:
prompt_v2 = f"""Translate this email into English.

Guidelines:
- Use British English
- Keep a calm and respectful tone
- Preserve the email format (greeting, body, sign-off)

{employee_email}
"""

messages = []
add_user_message(messages, prompt_v2)
response_v2 = chat(messages)
print(response_v2)


**Subject:** Leave Request

Dear [Manager's Name],

My name is John and I work at XYZ Company.
I would like to request leave to go on holiday.
Would it be possible to discuss which dates would suit the team and the company best?

Thank you in advance for your understanding, and I look forward to hearing from you.

Kind regards,
John


Better — the tone and format are now consistent. But the guidelines and the email text are just sitting next to each other as plain text. As a prompt grows (more guidelines, longer emails, multiple emails), Claude can start to lose track of where instructions end and data begins. XML tags fix that.


## 3. Structure with XML Tags

**Definition:** XML tags wrap distinct pieces of a prompt — like instructions versus data — in clear delimiters (e.g. `<email>...</email>`), so Claude can unambiguously tell them apart. This matters most once a prompt mixes instructions with a chunk of external content, exactly like our email here.

We wrap the French email in an `<email>` tag, separating it cleanly from the guidelines above it.


In [5]:
prompt_v3 = f"""Translate the email in the <email> tags into English.

Guidelines:
- Use British English
- Keep a calm and respectful tone
- Preserve the email format (greeting, body, sign-off)

<email>
{employee_email}
</email>
"""

messages = []
add_user_message(messages, prompt_v3)
response_v3 = chat(messages)
print(response_v3)


**Subject:** Leave Request

Dear [Manager's Name],

My name is John and I work at XYZ Ltd.

I would like to request leave to go on holiday. Would it be possible to discuss which dates would be most suitable for the team and the company?

Thank you in advance for your understanding, and I look forward to hearing from you.

Kind regards,
John


The output quality may look similar here since our email is short — but the XML boundary is what keeps larger prompts (longer documents, multiple emails, mixed instructions and data) from becoming ambiguous. Now let's add the last technique: showing Claude exactly what a great answer looks like.


## 4. Providing Examples (Few-Shot)

**Definition:** Few-shot prompting means showing Claude one or more worked examples of the task — a sample input paired with an ideal output — before giving it the real input. Examples communicate exactly the format, tone, and level of detail you want more precisely than instructions alone can.

We add a short `<example>` block: a different French email and the English translation we'd consider ideal for it. Claude can then match that pattern for John's email.


In [6]:
example_input = """
Objet : Absence pour rendez-vous médical

Bonjour,

Je dois m'absenter demain matin pour un rendez-vous médical.
Je serai de retour au bureau à partir de 14h.

Merci de votre compréhension,
Claire
"""

example_output = """Subject: Away for a Medical Appointment

Hello,

I need to be away tomorrow morning for a medical appointment.
I will be back in the office from 2pm.

Thank you for your understanding,
Claire
"""

prompt_v4 = f"""Translate the email in the <email> tags into English.

Guidelines:
- Use British English
- Keep a calm and respectful tone
- Preserve the email format (greeting, body, sign-off)

Here is an example of the kind of translation we want:

<example>
<input>
{example_input}
</input>
<ideal_output>
{example_output}
</ideal_output>
</example>

Now translate this email the same way:

<email>
{employee_email}
</email>
"""

messages = []
add_user_message(messages, prompt_v4)
response_v4 = chat(messages)
print(response_v4)


Subject: Leave Request

Dear [Manager's Name],

My name is John and I work at XYZ company.
I would like to request leave to go on holiday.
Would it be possible to discuss which dates would suit the team and the business best?

Thank you in advance for your understanding, and I look forward to hearing from you.

Kind regards,
John


## Summary

We answered the same request four times, adding one technique at each step:

| Step | Technique | What it added |
|---|---|---|
| v1 | Clear and Direct | A plain, unambiguous instruction |
| v2 | Being Specific | Concrete guidelines for tone, style, and format |
| v3 | XML Tags | A clear boundary between instructions and the email data |
| v4 | Providing Examples | A worked example showing exactly what a good answer looks like |

Each technique is additive — in practice you'll usually combine all of them, as we did by v4. Try changing `employee_email` to the manager's reply below and re-running `prompt_v4` to see the same prompt generalize to new input.


In [7]:
manager_reply = """
Subject: Re: Demande de Congé

Hi John,

Thank you for reaching out. I've reviewed your request for vacation leave. Please provide the specific dates you'd like to take off, so we can ensure proper coverage during your absence.

Looking forward to your response.

Best regards,
[Manager's Name]
"""

prompt_manager = f"""Translate the email in the <email> tags into French.

Guidelines:
- Use formal, polite French
- Preserve the email format (greeting, body, sign-off)

Here is an example of the kind of translation we want:

<example>
<input>
{example_output}
</input>
<ideal_output>
{example_input}
</ideal_output>
</example>

Now translate this email the same way:

<email>
{manager_reply}
</email>
"""

messages = []
add_user_message(messages, prompt_manager)
print(chat(messages))


Objet : Re: Demande de Congé

Bonjour John,

Merci de m'avoir contacté. J'ai examiné votre demande de congé. Pourriez-vous me communiquer les dates précises que vous souhaitez prendre, afin que nous puissions garantir une organisation adéquate pendant votre absence ?

Dans l'attente de votre réponse.

Cordialement,
[Nom du responsable]
